In [83]:
import os
from datetime import datetime
import importlib
import sys


from db_operations import (
    add_photo_by_filepath,
    check_photo_record_exists_by_filepath,
    update_photo_timestamp,
    delete_photo_by_filepath,
    get_photos_not_last_touched,
)

In [84]:
def _process_photo_files(directory, scan_timestamp):
    # Walk directory and insert or update each photo record.
    # Returns count of processed files.
    processed_files = 0

    for root, _, files in os.walk(directory):
        for scan_filename in files:
            scan_filename = os.path.join(root, scan_filename)
            print("Processing:", scan_filename)

            if check_photo_record_exists_by_filepath(scan_filename):
                update_photo_timestamp(scan_filename, scan_timestamp)
            else:
                add_photo_by_filepath(scan_filename, last_touched=scan_timestamp)

            processed_files += 1

    return processed_files


def _cleanup_outdated_photos(scan_timestamp):
    # Delete photo records that don't match the current scan timestamp.
    # Returns count of deleted records.
    outdated_photos = get_photos_not_last_touched(scan_timestamp)
    print("Outdated photos:", outdated_photos)

    deleted_count = 0
    for outdated_filepath in outdated_photos:
        print("Deleting:", outdated_filepath)
        delete_photo_by_filepath(outdated_filepath)
        deleted_count += 1

    return deleted_count


def sync_photos(directory="C:/Photos"):
    # Sync photos from directory: create new records, update existing ones, and clean up old records.
    scan_timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    if not os.path.isdir(directory):
        return {
            "scan_timestamp": scan_timestamp,
            "processed_files": 0,
            "deleted_records": 0,
            "message": f"Directory not found: {directory}",
        }

    processed_files = _process_photo_files(directory, scan_timestamp)
    deleted_count = _cleanup_outdated_photos(scan_timestamp)

    return {
        "scan_timestamp": scan_timestamp,
        "processed_files": processed_files,
        "deleted_records": deleted_count,
        "message": "Sync complete",
    }

In [86]:
sync_photos()

Processing: C:/Photos\v2ejgm0psffz.jpg
Processing: C:/Photos\vFapwnl.png
Processing: C:/Photos\vnc8fc10l1551.jpg
Processing: C:/Photos\vpoerfrsgap91.jpg
Processing: C:/Photos\w8k1zfwil7591.jpg
Processing: C:/Photos\what-do-you-guys-use-for-your-doorbell-messages-v0-qonypre2irdd1.webp
Processing: C:/Photos\wSFySgV.jpg
Processing: C:/Photos\New folder\r4CytJO - Copy - Copy.jpg
Outdated photos: ['C:/Photos\\New folder\\r4CytJO - Copy - Copy (2).jpg', 'C:/Photos\\New folder\\r4CytJO - Copy - Copy (3).jpg', 'C:/Photos\\New folder\\r4CytJO - Copy - Copy (4).jpg', 'C:/Photos\\New folder\\r4CytJO - Copy - Copy (5).jpg', 'C:/Photos\\New folder\\r4CytJO - Copy - Copy (6).jpg']
Deleting: C:/Photos\New folder\r4CytJO - Copy - Copy (2).jpg
Deleting: C:/Photos\New folder\r4CytJO - Copy - Copy (3).jpg
Deleting: C:/Photos\New folder\r4CytJO - Copy - Copy (4).jpg
Deleting: C:/Photos\New folder\r4CytJO - Copy - Copy (5).jpg
Deleting: C:/Photos\New folder\r4CytJO - Copy - Copy (6).jpg


{'scan_timestamp': '2026-05-03 16:45:35',
 'processed_files': 8,
 'deleted_records': 5,
 'message': 'Sync complete'}